# PINNs: 2차원 파동방정식  $u_{tt}=c^2(u_{xx}+u_{yy})$

원본 노트북의 뼈대(1번 점 → 2번 신경망 → 3번 손실 → 4번 최적화)는 그대로 두고, 아래 항목만 보강한 버전입니다.

- **상대 $L_2$ 오차 지표** 추가 (정확해 $u=\sin\pi x\,\sin\pi y\,\cos(\sqrt2\pi c t)$ 와 비교)
- **콜로케이션 점 확대 + 매 epoch 재샘플링** (옵션: Sobol 샘플링)
- **가중치는 설정 셀 한 곳에서만 정의**(4개), 고정 / gradient-norm 적응형 선택
- **학습 이력 저장**(손실 5종, 오차, 경과 시간) 및 로그 스케일 그래프
- **Adam(학습률 감쇠) → L-BFGS** 2단계 최적화
- 경계 4면 코드를 함수로 정리, `seed` 고정, `device`(GPU) 지원
- 시각화: 예측 / 정확해 / 절대오차 맵
- **[TM] PDE 내부 영역 Time-Marching** (`TM_MODE`로 선택): A `sequential`(시간 구간 분할) / B `causal`(인과성 가중치) / C `expand`(시간축 적응형 샘플링)  
  수정·추가된 부분은 코드에서 `[TM]` 주석으로 표시

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ============================================
# 0. 실험 설정 — 하이퍼파라미터는 전부 이 셀에서만 바꾼다
# ============================================
SEED = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

c = 1.0          # 파동 속도
T = 1.0          # 시간 영역 [0, T]

# --- 콜로케이션 점 ---
N_f, N_bc, N_ic = 5000, 1000, 1000   # PDE 내부 / 경계 전체 / 초기 조건 점 개수
RESAMPLE = True        # True: Adam 단계에서 매 epoch마다 점을 새로 뽑는다
SAMPLER = "random"     # "random" | "sobol"

# --- 손실 가중치 (여기서만 정의! 4개) ---
WEIGHTS_INIT = dict(w_pde=1.0, w_bc=1.0, w_ic=1.0, w_ic_velocity=1.0)
WEIGHT_MODE = "fixed"  # "fixed" | "grad_norm" (기울기 크기 기반 적응형)
W_UPDATE_EVERY = 100   # grad_norm 모드에서 가중치 갱신 주기(epoch)
W_ALPHA = 0.1          # 지수이동평균 계수

# --- 최적화 ---
EPOCHS_ADAM = 10000
LR_START, LR_END = 1e-3, 1e-5        # Adam 학습률: 지수 감쇠
USE_LBFGS = True                      #옵티마이저를 2개를 써서 최적화 시킴!
LBFGS_CHUNKS = 20                    # L-BFGS를 (chunks x iters) 만큼 수행
LBFGS_ITERS_PER_CHUNK = 50

# --- 평가 ---
EVAL_EVERY = 100       # 상대 L2 오차 계산 주기(epoch)
N_TEST = 20000         # 오차 평가용 고정 테스트 점 개수

# --- [TM] Time-Marching: PDE 내부 점(x_f, y_f, t_f)과 학습 루프의 시간축 처리 ---
#   "none"       : 기존 방식 (t_f ~ U[0, T], 시간축 처리 없음)
#   "sequential" : 방식 A — 시간 구간 분할, 앞 구간부터 단계적으로 학습
#   "causal"     : 방식 B — 인과성 가중치 Causal PINN (Wang, Sankaran & Perdikaris, 2022)
#   "expand"     : 방식 C — 시간축 적응형 샘플링 (학습 진행에 따라 t 범위를 0 → T 로 확장)
TM_MODE = "causal"

# [A] sequential — Adam epoch를 구간 수로 균등 분배 (EPOCHS_ADAM // TM_N_WINDOWS 씩)
TM_N_WINDOWS = 4          # 시간 구간 개수:  [0, T/4], [T/4, T/2], ... , [3T/4, T]
TM_REPLAY = 0.2           # 구간 k 학습 시 이전 구간 [0, T_{k-1}] 에 배치할 PDE 점의 비율 (망각 방지)
TM_SEQ_RESET_LR = True    # 구간이 바뀔 때 Adam 학습률을 LR_START로 재시작 (구간 안에서 LR_END까지 감쇠)

# [B] causal — w_i = exp(-eps * sum_{j<i} L_pde(t_j)),  L_pde = (1/N_t) sum_i w_i L_pde(t_i)
TM_CAUSAL_N_CHUNKS = 32                              # 시간축 bin 개수 N_t
TM_CAUSAL_EPS_LIST = [1e-2, 1e-1, 1.0, 10.0, 100.0]  # 인과성 계수 eps 스케줄 (원소 1개 = 고정 eps)
TM_CAUSAL_DELTA = 0.99                               # min_i w_i > delta 이면 다음 eps 로 증가

# [C] expand — t_f = t_max(epoch) * s^p,  s ~ U[0,1]
TM_EXPAND_T0 = 0.1        # 시작 시간 범위 비율 (t_max = 0.1*T 에서 시작)
TM_EXPAND_FRAC = 0.6      # Adam epoch의 이 비율 동안 t_max 를 T 까지 선형 확장 (이후는 [0, T] 전체)
TM_EXPAND_POWER = 2.0     # p 초기값 (>1 이면 t≈0 쪽에 점이 밀집), 확장이 끝날 때 p → 1 (균등)


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


set_seed(SEED)

# 1번: 데이터 포인트 입력(Collocation Points)

In [ ]:
# ============================================
# 1. 샘플링 함수
# ============================================
_sobol_calls = 0


def sample_uniform(n, dim):
    """[0,1]^dim 위의 균등 샘플 n개. SAMPLER="sobol"이면 scrambled Sobol (seed 고정)."""
    global _sobol_calls
    if SAMPLER == "sobol":
        _sobol_calls += 1
        engine = torch.quasirandom.SobolEngine(dim, scramble=True, seed=SEED + _sobol_calls)
        return engine.draw(n).to(device)
    return torch.rand(n, dim, device=device)


def make_boundary_points(N_bc):
    """단위 정사각형 경계 4면의 (x, y, t) 점. 면마다 N_bc // 4 개."""
    n = N_bc // 4
    zero = torch.zeros(n, 1, device=device)
    one = torch.ones(n, 1, device=device)
    faces = [sample_uniform(n, 2) for _ in range(4)]   # 각 면: (변하는 좌표, t)

    #            x = 0     x = 1     y = 0            y = 1
    x_bc = torch.cat([zero, one, faces[2][:, 0:1], faces[3][:, 0:1]])
    y_bc = torch.cat([faces[0][:, 0:1], faces[1][:, 0:1], zero, one])
    t_bc = torch.cat([f[:, 1:2] for f in faces]) * T
    return x_bc, y_bc, t_bc


# ============================================
# [TM] Time-Marching: PDE 내부 점의 시간 좌표 배치
# ============================================
TM_WINDOW_EDGES = [T * k / TM_N_WINDOWS for k in range(TM_N_WINDOWS + 1)]   # [A] 0 = T_0 < T_1 < ... < T_K = T
TM_EPOCHS_PER_WINDOW = max(1, EPOCHS_ADAM // TM_N_WINDOWS)                   # [A] 구간당 Adam epoch


def tm_window(epoch):
    """[A] epoch 가 속한 시간 구간 번호 k (0 ~ TM_N_WINDOWS-1). 나머지 epoch는 마지막 구간에 포함."""
    return min(epoch // TM_EPOCHS_PER_WINDOW, TM_N_WINDOWS - 1)


def tm_expand_state(epoch):
    """[C] 현재 시간 범위 t_max 와 밀집 지수 p."""
    prog = min(1.0, epoch / max(1, int(TM_EXPAND_FRAC * EPOCHS_ADAM)))   # 0 -> 1
    t_max = T * (TM_EXPAND_T0 + (1.0 - TM_EXPAND_T0) * prog)
    p = 1.0 + (TM_EXPAND_POWER - 1.0) * (1.0 - prog)
    return t_max, p


def map_time(s, epoch=None):
    """[0,1] 균등 샘플 s -> PDE 내부 점의 시간 좌표 t_f.
    s 를 그대로 사상하므로 SAMPLER="sobol" 의 3차원 저불일치성도 유지된다.
    epoch=None 이면 전체 [0, T] 균등 (초기 확인 / L-BFGS 단계)."""
    if epoch is None or TM_MODE in ("none", "causal"):
        return s * T            # [B] causal: 점은 균등 배치, 대신 손실에서 시간 bin 별 인과성 가중치 사용

    if TM_MODE == "sequential":   # [A]
        k = tm_window(epoch)
        t_lo, t_hi = TM_WINDOW_EDGES[k], TM_WINDOW_EDGES[k + 1]
        r = TM_REPLAY if k > 0 else 0.0
        if r <= 0.0:
            return t_lo + s * (t_hi - t_lo)
        # s < r  : 이미 학습한 구간 [0, t_lo] (replay, 비율 r)
        # s >= r : 현재 구간 [t_lo, t_hi]     (비율 1-r)
        return torch.where(s < r, s / r * t_lo, t_lo + (s - r) / (1.0 - r) * (t_hi - t_lo))

    if TM_MODE == "expand":       # [C]
        t_max, p = tm_expand_state(epoch)
        return t_max * s ** p

    raise ValueError(f"알 수 없는 TM_MODE: {TM_MODE}")


def tm_info(epoch):
    """로그 출력용: 현재 epoch 의 시간축 처리 상태."""
    if TM_MODE == "sequential":
        k = tm_window(epoch)
        return f"win {k + 1}/{TM_N_WINDOWS} [{TM_WINDOW_EDGES[k]:.2f},{TM_WINDOW_EDGES[k + 1]:.2f}]"
    if TM_MODE == "expand":
        t_max, p = tm_expand_state(epoch)
        return f"t_max {t_max:.3f} p {p:.2f}"
    if TM_MODE == "causal" and "tm_state" in globals() and tm_state["w"] is not None:
        return f"eps {TM_CAUSAL_EPS_LIST[tm_state['eps_idx']]:.0e} min_w {tm_state['w'].min().item():.3f}"
    return ""


def make_points(epoch=None):
    """PDE / 경계 / 초기 조건 점과 초기 조건 값을 한 번에 생성 (total_loss 인자 순서와 동일).
    [TM] epoch 를 넘기면 TM_MODE 에 따라 PDE 내부 점의 시간 범위/분포가 바뀐다. (BC, IC 점은 기존과 동일)"""
    # PDE 내부 영역 — [TM] t_f 는 map_time() 으로 배치
    xyz = sample_uniform(N_f, 3)
    x_f, y_f, t_f = xyz[:, 0:1], xyz[:, 1:2], map_time(xyz[:, 2:3], epoch)

    # 경계 조건 점
    x_bc, y_bc, t_bc = make_boundary_points(N_bc)

    # 초기 조건 점 (t = 0)
    xy = sample_uniform(N_ic, 2)
    x_ic, y_ic = xy[:, 0:1], xy[:, 1:2]

    # 초기 조건 값: 초기 변위 u, 초기 속도 u_t
    u_ic = torch.sin(torch.pi * x_ic) * torch.sin(torch.pi * y_ic)
    v_ic = torch.zeros_like(u_ic)

    return x_f, y_f, t_f, x_bc, y_bc, t_bc, x_ic, y_ic, u_ic, v_ic


pts = make_points()
print("점 shape:", [tuple(p.shape) for p in pts])

# **2번 심층 신경망 근사**

PINN 신경망 정의

In [ ]:
class PINN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 50),   # (x, y, t) 입력이니 3
            nn.Tanh(),          # 활성화 함수는 Tanh

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 1)    # 출력층
        )

    def forward(self, x, y, t):   # 순방향
        inputs = torch.cat([x, y, t], dim=1)
        return self.net(inputs)

PINNS 모델 생성 및 출력 확인  
(모델을 처음부터 다시 학습하고 싶으면 이 셀부터 다시 실행)

In [ ]:
set_seed(SEED)              # 같은 초기 가중치에서 시작하도록 고정
model = PINN().to(device)

u_pred = model(pts[0], pts[1], pts[2])
print(u_pred.shape)

# **3번 미분 연산 및 손실(Loss) 계산 모듈**

In [ ]:
# ============================================
# PDE Residual:  u_tt - c^2 (u_xx + u_yy)
# ============================================

def pde_residual(model, x, y, t, c=1.0):

    # 자동미분을 위해 입력값에 gradient 추적 설정
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)

    u = model(x, y, t)

    # 1차 미분
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    # 2차 미분
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_tt = torch.autograd.grad(u_t, t, grad_outputs=torch.ones_like(u_t), create_graph=True)[0]

    return u_tt - c**2 * (u_xx + u_yy)

개별 Loss 함수들

In [ ]:
def loss_pde(model, x_f, y_f, t_f, c=1.0):
    residual = pde_residual(model, x_f, y_f, t_f, c)
    return torch.mean(residual ** 2)


# Initial Displacement Loss:  u(x, y, 0) = u0(x, y)
def loss_ic(model, x_ic, y_ic, u_ic):
    t_ic = torch.zeros_like(x_ic)
    u_pred = model(x_ic, y_ic, t_ic)
    return torch.mean((u_pred - u_ic) ** 2)


# Initial Velocity Loss:  u_t(x, y, 0) = v0(x, y)
def loss_ic_velocity(model, x_ic, y_ic, v_ic):
    t_ic = torch.zeros_like(x_ic)
    t_ic.requires_grad_(True)

    u_pred = model(x_ic, y_ic, t_ic)
    u_t = torch.autograd.grad(u_pred, t_ic, grad_outputs=torch.ones_like(u_pred),
                              create_graph=True)[0]
    return torch.mean((u_t - v_ic) ** 2)


# Boundary Condition Loss:  u = 0 (동차 디리클레)
def loss_bc(model, x_bc, y_bc, t_bc):
    u_pred = model(x_bc, y_bc, t_bc)
    return torch.mean(u_pred ** 2)

Total

In [ ]:
def total_loss(
    model,
    x_f, y_f, t_f,
    x_bc, y_bc, t_bc,
    x_ic, y_ic,
    u_ic,
    v_ic,
    c=1.0,
    w_pde=1.0,
    w_bc=1.0,
    w_ic=1.0,
    w_ic_velocity=1.0
):

    L_pde = loss_pde(model, x_f, y_f, t_f, c)
    L_bc = loss_bc(model, x_bc, y_bc, t_bc)
    L_ic = loss_ic(model, x_ic, y_ic, u_ic)

    # 시간에 대해 2계 방정식이므로 초기 변위 u 와 초기 속도 u_t 두 조건을 모두 맞춰야 한다
    L_ic_velocity = loss_ic_velocity(model, x_ic, y_ic, v_ic)

    L_loss = (
        w_pde * L_pde
        + w_bc * L_bc
        + w_ic * L_ic
        + w_ic_velocity * L_ic_velocity
    )

    return L_loss, L_pde, L_bc, L_ic, L_ic_velocity


# 사용법:  total_loss(model, *pts, c=c, **weights)

[TM] Time-Marching 손실 래퍼 `total_loss_tm`
기존 `total_loss`/개별 손실 함수는 그대로 두고, 학습 루프에서는 이 래퍼를 호출합니다.

- `TM_MODE != "causal"` : `total_loss` 를 그대로 호출 (A, C 는 점 배치만 바뀜)
- `TM_MODE == "causal"` : PDE 항만 인과성 가중 손실로 교체 (BC / IC 항은 기존 함수 그대로)

$$\mathcal L_{pde}=\frac1{N_t}\sum_{i=1}^{N_t} w_i\,\mathcal L_{pde}(t_i),\qquad
w_i=\exp\Big(-\varepsilon\sum_{j<i}\mathcal L_{pde}(t_j)\Big)\ (\text{gradient 차단})$$

$\mathcal L_{pde}(t_i)$ 는 $t\in[t_{i-1},t_i)$ bin 에 속한 점들의 잔차 제곱 평균.
$\min_i w_i>\delta$ (= 전 시간대가 충분히 학습됨) 이면 $\varepsilon$ 을 다음 값으로 키웁니다.

In [ ]:
# ============================================
# [TM] 방식 B: 인과성 가중 PDE 손실 + 공통 래퍼
# ============================================
tm_state = {"eps_idx": 0,      # 현재 eps = TM_CAUSAL_EPS_LIST[eps_idx]
            "w": None,         # 마지막으로 계산된 인과성 가중치 (N_t,)
            "L_t": None,       # 마지막으로 계산된 bin 별 PDE 손실 (N_t,)
            "w_frozen": None}  # L-BFGS 단계에서 고정할 가중치 (None 이면 매번 새로 계산)


def causal_pde_loss(model, x_f, y_f, t_f, c=1.0, eps=1.0, w_fixed=None):
    """시간 bin 별 PDE 손실 L_t 와 인과성 가중치 w 로 가중 평균한 PDE 손실."""
    r2 = pde_residual(model, x_f, y_f, t_f, c).pow(2).flatten()
    Nt = TM_CAUSAL_N_CHUNKS
    idx = torch.clamp((t_f.detach().flatten() / T * Nt).long(), 0, Nt - 1)       # 각 점의 시간 bin
    sums = torch.zeros(Nt, device=r2.device, dtype=r2.dtype).index_add(0, idx, r2)
    cnt = torch.bincount(idx, minlength=Nt).clamp_min(1).to(r2.dtype)
    L_t = sums / cnt                                                              # (N_t,) bin 별 평균

    if w_fixed is None:
        with torch.no_grad():                         # 가중치는 상수 취급 (gradient 차단)
            cum = torch.cumsum(L_t, 0) - L_t          # sum_{j<i} L_t[j]
            w = torch.exp(-eps * cum)
    else:
        w = w_fixed
    return torch.mean(w * L_t), L_t.detach(), w


def total_loss_tm(model, x_f, y_f, t_f, x_bc, y_bc, t_bc, x_ic, y_ic, u_ic, v_ic,
                  c=1.0, w_pde=1.0, w_bc=1.0, w_ic=1.0, w_ic_velocity=1.0):
    """total_loss 와 같은 인자/반환값. causal 모드에서만 PDE 항을 인과성 가중 손실로 교체."""
    if TM_MODE != "causal":
        return total_loss(model, x_f, y_f, t_f, x_bc, y_bc, t_bc, x_ic, y_ic, u_ic, v_ic,
                          c=c, w_pde=w_pde, w_bc=w_bc, w_ic=w_ic, w_ic_velocity=w_ic_velocity)

    eps = TM_CAUSAL_EPS_LIST[tm_state["eps_idx"]]
    L_pde, L_t, w = causal_pde_loss(model, x_f, y_f, t_f, c, eps, tm_state["w_frozen"])
    tm_state["w"], tm_state["L_t"] = w.detach(), L_t

    L_bc = loss_bc(model, x_bc, y_bc, t_bc)
    L_ic = loss_ic(model, x_ic, y_ic, u_ic)
    L_ic_velocity = loss_ic_velocity(model, x_ic, y_ic, v_ic)

    L_loss = w_pde * L_pde + w_bc * L_bc + w_ic * L_ic + w_ic_velocity * L_ic_velocity
    return L_loss, L_pde, L_bc, L_ic, L_ic_velocity


def tm_causal_anneal():
    """min_i w_i > delta 이면 eps 를 다음 단계로 올린다. 올렸으면 True."""
    if tm_state["w"] is None or tm_state["eps_idx"] >= len(TM_CAUSAL_EPS_LIST) - 1:
        return False
    if tm_state["w"].min().item() > TM_CAUSAL_DELTA:
        tm_state["eps_idx"] += 1
        return True
    return False


def tm_freeze_causal_weights(pts):
    """L-BFGS 용: 현재 점/모델에서 인과성 가중치를 계산해 고정 (line search 중 목적함수가 변하지 않도록)."""
    tm_state["w_frozen"] = None
    eps = TM_CAUSAL_EPS_LIST[tm_state["eps_idx"]]
    _, _, w = causal_pde_loss(model, pts[0], pts[1], pts[2], c, eps)
    tm_state["w_frozen"] = w.detach()
    tm_state["w"] = tm_state["w_frozen"]


# 사용법:  total_loss_tm(model, *pts, c=c, **weights)


가중치 적응형 갱신 (선택: `WEIGHT_MODE = "grad_norm"`)  
Wang, Teng & Perdikaris (2021)의 기울기 균형 아이디어를 단순화한 버전입니다:
$\hat w_i = \max|\nabla_\theta L_{pde}| \,/\, \mathrm{mean}|\nabla_\theta L_i|$, 이동평균으로 갱신.

In [ ]:
def update_weights_gradnorm(model, parts, weights, alpha=0.1):
    """parts = (L_pde, L_bc, L_ic, L_ic_velocity). weights(dict)를 제자리에서 갱신."""
    params = [p for p in model.parameters() if p.requires_grad]

    def grad_vec(L):
        gs = torch.autograd.grad(L, params, retain_graph=True, allow_unused=True)
        return torch.cat([g.flatten() for g in gs if g is not None])

    L_pde, L_bc, L_ic, L_icv = parts
    max_pde = grad_vec(L_pde).abs().max()

    for key, L in (("w_bc", L_bc), ("w_ic", L_ic), ("w_ic_velocity", L_icv)):
        mean_i = grad_vec(L).abs().mean()
        target = (max_pde / (mean_i + 1e-8)).item()
        weights[key] = (1 - alpha) * weights[key] + alpha * target
    # w_pde는 1로 고정 (기준)

## 정확도 지표 (상대 $L_2$ 오차)

정확해 (이 문제의 초기·경계 조건에서): $u(x,y,t)=\sin(\pi x)\sin(\pi y)\cos(\sqrt{2}\,\pi c\,t)$  
※ 초기 조건 / 경계 조건 / $c$ 를 바꾸면 정확해도 함께 바꿔야 합니다.

In [ ]:
def exact(x, y, t, c=1.0):
    return torch.sin(torch.pi * x) * torch.sin(torch.pi * y) * torch.cos(math.sqrt(2) * torch.pi * c * t)


# 평가용 테스트 점은 한 번만 뽑아 고정 (학습용 난수와 분리 -> 실험 간 공정 비교)
_g = torch.Generator(device=device)
_g.manual_seed(1234)
x_test = torch.rand(N_TEST, 1, device=device, generator=_g)
y_test = torch.rand(N_TEST, 1, device=device, generator=_g)
t_test = torch.rand(N_TEST, 1, device=device, generator=_g) * T
u_test = exact(x_test, y_test, t_test, c)


def rel_l2(model):
    with torch.no_grad():
        u_hat = model(x_test, y_test, t_test)
        return (torch.linalg.norm(u_hat - u_test) / torch.linalg.norm(u_test)).item()


print("초기(학습 전) 상대 L2 오차:", rel_l2(model))

# **4번 최종 손실 합산 및 최적화**

In [ ]:
# ============================================
# 4. Optimization 설정 + 학습 이력 저장소
# ============================================
weights = dict(WEIGHTS_INIT)     # 가중치는 WEIGHTS_INIT 한 곳에서만 정의됨

optimizer = torch.optim.Adam(model.parameters(), lr=LR_START)
gamma = (LR_END / LR_START) ** (1.0 / EPOCHS_ADAM)          # 마지막 epoch에서 lr = LR_END
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

history = {"total": [], "pde": [], "bc": [], "ic": [], "ic_v": []}   # 매 step
history_err = {"step": [], "rel_l2": [], "time": []}                 # EVAL_EVERY마다
train_time = 0.0            # 누적 학습 시간 [s] (오차 평가 시간은 제외)
stage_boundary = None       # Adam -> L-BFGS 전환 step


def record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity):
    history["total"].append(L_loss.item())
    history["pde"].append(L_pde.item())
    history["bc"].append(L_bc.item())
    history["ic"].append(L_ic.item())
    history["ic_v"].append(L_ic_velocity.item())


def record_error():
    history_err["step"].append(len(history["total"]))
    history_err["rel_l2"].append(rel_l2(model))
    history_err["time"].append(train_time)

실제 학습 반복 — 1단계: Adam

In [ ]:
for epoch in range(EPOCHS_ADAM):

    # [TM-A] 새 시간 구간 시작: 안내 출력 + (옵션) 학습률 재시작
    new_window = (TM_MODE == "sequential" and epoch % TM_EPOCHS_PER_WINDOW == 0
                  and epoch // TM_EPOCHS_PER_WINDOW < TM_N_WINDOWS)
    if new_window:
        k = tm_window(epoch)
        print(f"--- [TM-A] 구간 {k + 1}/{TM_N_WINDOWS}: t in [{TM_WINDOW_EDGES[k]:.3f}, {TM_WINDOW_EDGES[k + 1]:.3f}]"
              f"  (replay {TM_REPLAY if k > 0 else 0.0:.0%} from [0, {TM_WINDOW_EDGES[k]:.3f}]) ---")
        if TM_SEQ_RESET_LR:
            n_ep = TM_EPOCHS_PER_WINDOW if k < TM_N_WINDOWS - 1 else EPOCHS_ADAM - k * TM_EPOCHS_PER_WINDOW
            for g in optimizer.param_groups:
                g["lr"] = LR_START
            scheduler = torch.optim.lr_scheduler.ExponentialLR(
                optimizer, gamma=(LR_END / LR_START) ** (1.0 / n_ep))

    t_start = time.time()

    # [TM] RESAMPLE=False 여도 시간 범위가 바뀌는 순간(A: 구간 전환, C: 매 epoch)에는 점을 새로 뽑는다
    if RESAMPLE or new_window or TM_MODE == "expand":
        pts = make_points(epoch)     # [TM] epoch 에 따라 PDE 내부 점의 시간 범위/분포 결정

    optimizer.zero_grad()

    L_loss, L_pde, L_bc, L_ic, L_ic_velocity = total_loss_tm(model, *pts, c=c, **weights)   # [TM]

    # (선택) 기울기 크기 기반 가중치 갱신 — backward 전에 수행, 새 가중치는 다음 epoch부터 적용
    if WEIGHT_MODE == "grad_norm" and epoch % W_UPDATE_EVERY == 0:
        update_weights_gradnorm(model, (L_pde, L_bc, L_ic, L_ic_velocity), weights, W_ALPHA)

    L_loss.backward()
    optimizer.step()
    scheduler.step()

    # [TM-B] 모든 시간 bin 의 가중치가 delta 를 넘으면 eps 증가 (더 엄격한 인과성)
    if TM_MODE == "causal" and tm_causal_anneal():
        print(f"    [TM-B] epoch {epoch}: min w > {TM_CAUSAL_DELTA} -> eps = "
              f"{TM_CAUSAL_EPS_LIST[tm_state['eps_idx']]:g}")

    record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity)
    train_time += time.time() - t_start

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS_ADAM - 1:
        record_error()

    if epoch % 500 == 0 or epoch == EPOCHS_ADAM - 1:
        print(
            f"Epoch {epoch:5d} | "
            f"Total: {L_loss.item():.3e} | "
            f"PDE: {L_pde.item():.3e} | "
            f"BC: {L_bc.item():.3e} | "
            f"IC: {L_ic.item():.3e} | "
            f"IC_v: {L_ic_velocity.item():.3e} | "
            f"relL2: {history_err['rel_l2'][-1]:.3e} | "
            f"time: {train_time:.1f}s | "
            f"{tm_info(epoch)}"                       # [TM] 시간축 처리 상태
        )

stage_boundary = len(history["total"])
print("\n현재 가중치:", {k: round(v, 3) for k, v in weights.items()})


2단계: L-BFGS 정밀화  
(L-BFGS는 같은 점에서 결정적으로 평가해야 하므로 점을 한 번 새로 뽑아 고정하고, 가중치도 고정)  
[TM] Adam 단계의 Time-Marching 이 끝난 뒤이므로 PDE 점은 전체 $[0,T]$ 에서 뽑습니다(`make_points()`).  
causal 모드는 chunk 마다 인과성 가중치를 다시 계산해 **chunk 안에서는 고정**합니다(line search 안정성).

In [ ]:
if USE_LBFGS:
    pts = make_points()               # 고정 점 ([TM] epoch=None -> 전체 [0, T])

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=LBFGS_ITERS_PER_CHUNK,
        history_size=50,
        tolerance_grad=1e-10,
        tolerance_change=1e-14,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad()
        L_loss, L_pde, L_bc, L_ic, L_ic_velocity = total_loss_tm(model, *pts, c=c, **weights)   # [TM]
        L_loss.backward()
        record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity)   # 함수 평가 1회 = 1 step
        return L_loss

    for chunk in range(LBFGS_CHUNKS):
        t_start = time.time()
        if TM_MODE == "causal":           # [TM-B] chunk 시작 시 가중치 재계산 후 고정
            tm_freeze_causal_weights(pts)
        lbfgs.step(closure)
        train_time += time.time() - t_start

        record_error()
        print(
            f"L-BFGS chunk {chunk + 1:2d}/{LBFGS_CHUNKS} | "
            f"Total: {history['total'][-1]:.3e} | "
            f"relL2: {history_err['rel_l2'][-1]:.3e} | "
            f"time: {train_time:.1f}s | "
            f"{tm_info(EPOCHS_ADAM) if TM_MODE == 'causal' else ''}"
        )
        if not math.isfinite(history["total"][-1]):
            print("손실이 발산(NaN/inf)하여 L-BFGS를 중단합니다.")
            break
        if TM_MODE == "causal" and tm_causal_anneal():     # [TM-B] 다음 chunk 부터 더 큰 eps
            print(f"    [TM-B] eps = {TM_CAUSAL_EPS_LIST[tm_state['eps_idx']]:g}")
    tm_state["w_frozen"] = None
else:
    print("USE_LBFGS=False: L-BFGS 단계를 건너뜁니다.")

# **5번 결과 평가 및 시각화**

In [ ]:
model.eval()

final_err = rel_l2(model)
adam_end = [e for s, e in zip(history_err["step"], history_err["rel_l2"]) if s == stage_boundary]

with torch.no_grad():
    bx, by, bt = make_boundary_points(4000)
    max_bc = model(bx, by, bt).abs().max().item()     # 정확해는 경계에서 0

print(f"최종 상대 L2 오차          : {final_err:.4e}")
if adam_end:
    print(f"(Adam 종료 시점 상대 L2 오차: {adam_end[0]:.4e})")
print(f"경계에서 max |u|            : {max_bc:.4e}  (정확해는 0)")
print(f"총 학습 시간 (평가 제외)   : {train_time:.1f} s")
print(f"설정: N_f={N_f}, N_bc={N_bc}, N_ic={N_ic}, RESAMPLE={RESAMPLE}, SAMPLER={SAMPLER}, "
      f"WEIGHT_MODE={WEIGHT_MODE}, Adam={EPOCHS_ADAM}, L-BFGS={USE_LBFGS}, TM_MODE={TM_MODE}")

학습 이력 (로그 스케일)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

steps = np.arange(1, len(history["total"]) + 1)
for key, label in [("total", "Total (weighted)"), ("pde", "PDE"), ("bc", "BC"),
                   ("ic", "IC (u)"), ("ic_v", "IC (u_t)")]:
    ax[0].semilogy(steps, history[key], label=label, lw=1)
ax[0].set_title("Loss history")
ax[0].set_xlabel("step (Adam epoch / L-BFGS function eval)")
ax[0].legend()

ax[1].semilogy(history_err["step"], history_err["rel_l2"], "o-", ms=2)
ax[1].set_title("Relative L2 error vs step")
ax[1].set_xlabel("step")

ax[2].semilogy(history_err["time"], history_err["rel_l2"], "o-", ms=2)
ax[2].set_title("Relative L2 error vs training time")
ax[2].set_xlabel("training time [s] (evaluation excluded)")

if USE_LBFGS and stage_boundary is not None:
    for a in ax[:2]:
        a.axvline(stage_boundary, ls="--", color="gray")

for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

예측 / 정확해 / 절대오차 맵

In [ ]:
n = 100
xs = torch.linspace(0, 1, n, device=device)
Xg, Yg = torch.meshgrid(xs, xs, indexing="xy")
X_flat, Y_flat = Xg.reshape(-1, 1), Yg.reshape(-1, 1)
X_np, Y_np = Xg.cpu().numpy(), Yg.cpu().numpy()

times = [0.0, 0.5 * T, T]     # [TM] T 를 바꿔도 시작/중간/끝 시각을 그림
fig, axes = plt.subplots(3, len(times), figsize=(15, 12))

for j, t_val in enumerate(times):
    T_flat = torch.full_like(X_flat, t_val)

    with torch.no_grad():
        U_pred = model(X_flat, Y_flat, T_flat)
        U_true = exact(X_flat, Y_flat, T_flat, c)

    rel = (torch.linalg.norm(U_pred - U_true) / torch.linalg.norm(U_true)).item()

    P = U_pred.reshape(n, n).cpu().numpy()
    E = U_true.reshape(n, n).cpu().numpy()
    A = np.abs(P - E)

    # 예측과 정확해는 같은 색 범위를 사용해야 비교가 된다
    vmin, vmax = min(P.min(), E.min()), max(P.max(), E.max())
    levels = np.linspace(vmin, vmax, 51)

    cf0 = axes[0, j].contourf(X_np, Y_np, P, levels=levels, cmap="viridis")
    axes[0, j].set_title(f"Prediction, t = {t_val}")
    fig.colorbar(cf0, ax=axes[0, j])

    cf1 = axes[1, j].contourf(X_np, Y_np, E, levels=levels, cmap="viridis")
    axes[1, j].set_title(f"Exact, t = {t_val}")
    fig.colorbar(cf1, ax=axes[1, j])

    cf2 = axes[2, j].contourf(X_np, Y_np, A, levels=50, cmap="magma")
    axes[2, j].set_title(f"|Error|, t = {t_val}  (rel L2 = {rel:.2e})")
    fig.colorbar(cf2, ax=axes[2, j])

    for i in range(3):
        axes[i, j].set_xlabel("x")
        axes[i, j].set_ylabel("y")

plt.tight_layout()
plt.show()

[TM] 시간별 상대 $L_2$ 오차 (Time-Marching 효과 확인)
각 시각 $t$ 에서 공간 격자 위 상대 오차를 계산합니다. 인과성이 잘 지켜졌다면 $t$ 가 커져도 오차가 급격히 커지지 않습니다.
causal 모드면 마지막 인과성 가중치 $w_i$ 와 bin 별 PDE 손실도 함께 그립니다.

In [ ]:
# [TM] 시간별 상대 L2 오차
t_line = np.linspace(0.0, T, 51)
err_t = []
with torch.no_grad():
    for t_val in t_line:
        T_flat = torch.full_like(X_flat, float(t_val))
        U_pred = model(X_flat, Y_flat, T_flat)
        U_true = exact(X_flat, Y_flat, T_flat, c)
        err_t.append((torch.linalg.norm(U_pred - U_true) / (torch.linalg.norm(U_true) + 1e-12)).item())
# 주의: cos(sqrt2*pi*c*t) = 0 근처에서는 정확해 크기가 0 에 가까워 상대 오차가 커 보일 수 있다

n_ax = 2 if (TM_MODE == "causal" and tm_state["w"] is not None) else 1
fig, ax = plt.subplots(1, n_ax, figsize=(6.5 * n_ax, 4), squeeze=False)
ax = ax[0]
ax[0].semilogy(t_line, err_t, "o-", ms=3)
if TM_MODE == "sequential":
    for e in TM_WINDOW_EDGES[1:-1]:
        ax[0].axvline(e, ls="--", color="gray")
ax[0].set_title(f"Relative L2 error vs t  (TM_MODE={TM_MODE})")
ax[0].set_xlabel("t")
ax[0].grid(alpha=0.3)

if n_ax == 2:
    centers = (np.arange(TM_CAUSAL_N_CHUNKS) + 0.5) / TM_CAUSAL_N_CHUNKS * T
    ax[1].plot(centers, tm_state["w"].cpu().numpy(), "o-", ms=3, label="causal weight w_i")
    ax1b = ax[1].twinx()
    ax1b.semilogy(centers, tm_state["L_t"].cpu().numpy(), "s-", ms=3, color="tab:red", label="L_pde(t_i)")
    ax[1].set_title(f"causal weights (eps={TM_CAUSAL_EPS_LIST[tm_state['eps_idx']]:g})")
    ax[1].set_xlabel("t")
    ax[1].set_ylim(-0.05, 1.05)
    ax[1].legend(loc="lower left")
    ax1b.legend(loc="upper right")
    ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
torch.save({"state_dict": model.state_dict(), "train_time": train_time}, "pinn_model.pt")